# Gemma Fine-Tuning Pipeline for URA Tax Assistant

This notebook implements an optimized fine-tuning pipeline using **Unsloth** and **QLoRA** for
training a Gemma-2 language model on Uganda Revenue Authority (URA) tax domain data.

### Pipeline Overview

```
Installation -> Imports -> Config -> Model Loading -> Data Loading ->
Tokenization -> Training -> Evaluation -> Export -> Report
```

### Features

- **Unsloth**: 2x faster training with memory-efficient optimizations
- **QLoRA**: 4-bit quantization with LoRA adapters for parameter efficiency
- **IEEE Visualizations**: Publication-quality charts and metrics
- **Adaptive Training**: Auto-detects pre-trained models for fine-tuning mode
- **CI/CD Integration**: Outputs formatted for Kaggle and GitHub Actions pipelines

In [ ]:
# Cell 1: Installation & Setup with Visualization Libraries
%%capture
# Install all required packages with optimized versions
!pip install --upgrade --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade --quiet transformers datasets accelerate peft bitsandbytes trl
!pip install --upgrade --quiet flash-attn --no-build-isolation
!pip install --upgrade --quiet sentencepiece protobuf einops

# Install Unsloth for 2x faster training
!pip install --upgrade --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# For data processing and PDF extraction (optional)
#!pip install --upgrade --quiet pymupdf4llm python-docx pandas numpy

# Install visualization libraries for IEEE publications
!pip install --upgrade --quiet matplotlib seaborn plotly scienceplots scikit-learn
!pip install --upgrade --quiet kaleido  # For plotly export

# Install wandb for logging (optional)
!pip install --upgrade --quiet wandb

# Clear cache to free up memory
import gc
gc.collect()

## 1. Imports and Environment Setup

Import all required libraries and configure the environment for
reproducible training with IEEE publication-quality visualizations.

In [ ]:
# Cell 2: Import Libraries & Setup with IEEE Styling
import os
import json
import random
import warnings
import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
import numpy as np
import pandas as pd

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter

# Apply IEEE publication style for matplotlib
try:
    plt.style.use(['science', 'ieee', 'no-latex'])
    print("✅ IEEE publication style applied to matplotlib")
except (OSError, ValueError):
    # SciencePlots not available; fall back to seaborn whitegrid
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted')
    print("SciencePlots not available, using seaborn whitegrid style")

# Configure plotly for IEEE
plotly_config = {
    'displayModeBar': False,
    'responsive': True
}

# Set environment variables for optimization
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Suppress warnings
warnings.filterwarnings("ignore")

# Create visualization output directory
VIZ_DIR = "/kaggle/working/visualizations"
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"✅ Visualization directory created: {VIZ_DIR}")

# Check GPU information
import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ CUDA Version: {torch.version.cuda}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\  No GPU detected. Training will be very slow!")

## 2. Configuration and Hyperparameters

All training hyperparameters and visualization settings are centralized here.
Select a model target from `MODEL_CONFIGS` to switch between architectures.

| Parameter | Gemma-2 2B | Llama-3 1B | TinyLlama |
|-----------|-----------|-----------|-----------|
| LoRA Rank | 32 | 16 | 16 |
| Epochs | 3 | 4 | 5 |
| Batch Size | 4 | 8 | 16 |
| Max Seq Length | 2048 | 1024 | 512 |

In [ ]:
# Cell 3: Configuration & Hyperparameters with Visualization Config
# IEEE Color palette for consistent styling
IEEE_COLORS = {
    'blue': '#0066CC',
    'red': '#CC3333',
    'green': '#339933',
    'orange': '#FF9900',
    'purple': '#9966CC',
    'gray': '#666666'
}

# Configuration for different model targets
MODEL_CONFIGS = {
    "gemma2-2b": {
        "model_id": "unsloth/gemma-2-2b",
        "max_seq_length": 2048,
        "lora_r": 32,
        "lora_alpha": 64,
        "epochs": 3,
        "learning_rate": 2e-4,
        "batch_size": 4,
        "gradient_accumulation": 4,
        "color": IEEE_COLORS['blue']
    },
    "llama3-1b": {
        "model_id": "unsloth/llama-3-2-1b-bnb-4bit",
        "max_seq_length": 1024,
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 4,
        "learning_rate": 2e-4,
        "batch_size": 8,
        "gradient_accumulation": 2,
        "color": IEEE_COLORS['red']
    },
    "tinyllama": {
        "model_id": "unsloth/tinyllama-bnb-4bit",
        "max_seq_length": 512,
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 5,
        "learning_rate": 3e-4,
        "batch_size": 16,
        "gradient_accumulation": 1,
        "color": IEEE_COLORS['green']
    }
}

# Visualization configuration
class VisualizationConfig:
    # Figure sizes for IEEE publications
    SINGLE_COLUMN_WIDTH = 3.5  # inches (IEEE single column)
    DOUBLE_COLUMN_WIDTH = 7.2  # inches (IEEE double column)
    GOLDEN_RATIO = 1.618
    
    # Font sizes
    TITLE_FONT_SIZE = 10
    AXIS_FONT_SIZE = 9
    TICK_FONT_SIZE = 8
    LEGEND_FONT_SIZE = 8
    
    # Line styles
    LINE_WIDTH = 1.5
    MARKER_SIZE = 4
    
    # Save formats
    SAVE_FORMATS = ['png', 'pdf', 'svg']  # PDF for IEEE publications
    
    # Color cycle
    COLOR_CYCLE = list(IEEE_COLORS.values())

# Training configuration
class TrainingConfig:
    # Model selection (choose from MODEL_CONFIGS keys)
    MODEL_TARGET = "gemma2-2b"  # or "llama3-1b" or "tinyllama"
    
    # Visualization settings
    SAVE_VISUALIZATIONS = True
    GENERATE_IEEEFIGURES = True
    
    # Check for pre-trained model (fine-tuning mode)
    PRETRAINED_MODEL_PATH = "/kaggle/input/finetune_dataset/pretrained_model"
    IS_FINETUNING = os.path.exists(PRETRAINED_MODEL_PATH) and any(
        f.endswith(('.bin', '.safetensors')) for f in os.listdir(PRETRAINED_MODEL_PATH)
    )
    
    # Adjust training parameters based on mode
    if IS_FINETUNING:
        # Fine-tuning parameters (lower learning rate, fewer epochs)
        NUM_EPOCHS = 2  # Fewer epochs for fine-tuning
        LEARNING_RATE = 1e-5  # Lower learning rate for fine-tuning
        LORA_R = 16  # Smaller LoRA rank for fine-tuning
        LORA_ALPHA = 32  # Adjusted alpha
    else:
        # Initial training parameters
        NUM_EPOCHS = MODEL_CONFIGS[MODEL_TARGET]["epochs"]
        LEARNING_RATE = MODEL_CONFIGS[MODEL_TARGET]["learning_rate"]
        LORA_R = MODEL_CONFIGS[MODEL_TARGET]["lora_r"]
        LORA_ALPHA = MODEL_CONFIGS[MODEL_TARGET]["lora_alpha"]
    
    # Common parameters
    MAX_SEQ_LENGTH = MODEL_CONFIGS[MODEL_TARGET]["max_seq_length"]
    BATCH_SIZE = MODEL_CONFIGS[MODEL_TARGET]["batch_size"]
    GRADIENT_ACCUMULATION = MODEL_CONFIGS[MODEL_TARGET]["gradient_accumulation"]
    LORA_DROPOUT = 0.05
    
    # Optimization
    USE_GRADIENT_CHECKPOINTING = True
    USE_FLASH_ATTENTION_2 = True
    WARMUP_STEPS = 50
    MAX_GRAD_NORM = 0.3
    WEIGHT_DECAY = 0.01
    
    # Data
    TRAIN_TEST_SPLIT = 0.1
    DATASET_SHUFFLE = True
    PACKING = False
    
    # Logging
    LOGGING_STEPS = 10
    SAVE_STEPS = 100
    EVAL_STEPS = 100
    
    # Output
    OUTPUT_DIR = "/kaggle/working/finetuned_model"
    SAVE_TOTAL_LIMIT = 3

# Print configuration with visual formatting
config = MODEL_CONFIGS[TrainingConfig.MODEL_TARGET]
print("⚙️  TRAINING CONFIGURATION")
print("="*50)
print(f"{'Mode:':<25} {'Fine-tuning' if TrainingConfig.IS_FINETUNING else 'Initial Training'}")
print(f"{'Model:':<25} {config['model_id']}")
print(f"{'Target:':<25} {TrainingConfig.MODEL_TARGET}")
print(f"{'Max Sequence Length:':<25} {TrainingConfig.MAX_SEQ_LENGTH}")
print(f"{'Batch Size:':<25} {TrainingConfig.BATCH_SIZE}")
print(f"{'Gradient Accumulation:':<25} {TrainingConfig.GRADIENT_ACCUMULATION}")
print(f"{'Effective Batch Size:':<25} {TrainingConfig.BATCH_SIZE * TrainingConfig.GRADIENT_ACCUMULATION}")
print(f"{'Epochs:':<25} {TrainingConfig.NUM_EPOCHS}")
print(f"{'Learning Rate:':<25} {TrainingConfig.LEARNING_RATE:.2e}")
print(f"{'LoRA Rank (r):':<25} {TrainingConfig.LORA_R}")
print(f"{'LoRA Alpha:':<25} {TrainingConfig.LORA_ALPHA}")
if TrainingConfig.IS_FINETUNING:
    print(f"{'Pre-trained Model:':<25} ✅ Loaded from {TrainingConfig.PRETRAINED_MODEL_PATH}")
print("="*50)

## 3. Visualization Utilities

IEEE publication-quality visualization utilities for training curves,
parameter efficiency plots, and interactive dashboards.

In [ ]:
# Cell 4: Visualization Utilities for IEEE Publications
class IEEEVisualizer:
    """Class for creating IEEE publication quality visualizations"""
    
    @staticmethod
    def create_figure(figsize=None, dpi=300, nrows=1, ncols=1):
        """Create matplotlib figure with IEEE styling"""
        if figsize is None:
            figsize = (VisualizationConfig.SINGLE_COLUMN_WIDTH, 
                      VisualizationConfig.SINGLE_COLUMN_WIDTH / VisualizationConfig.GOLDEN_RATIO)
        
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=dpi, 
                                 constrained_layout=True)
        
        if nrows == 1 and ncols == 1:
            axes = np.array([axes])
        
        # Apply IEEE styling
        for ax in axes.flat:
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', which='major', 
                          labelsize=VisualizationConfig.TICK_FONT_SIZE)
        
        return fig, axes
    
    @staticmethod
    def save_figure(fig, filename, formats=None, dpi=300):
        """Save figure in multiple formats"""
        if formats is None:
            formats = VisualizationConfig.SAVE_FORMATS
        
        saved_files = []
        for fmt in formats:
            filepath = os.path.join(VIZ_DIR, f"{filename}.{fmt}")
            fig.savefig(filepath, format=fmt, dpi=dpi, bbox_inches='tight')
            saved_files.append(filepath)
        
        return saved_files
    
    @staticmethod
    def create_training_curves_plot(train_losses, val_losses, train_steps=None):
        """Create training curves plot for IEEE publications"""
        fig, ax = IEEEVisualizer.create_figure(
            figsize=(VisualizationConfig.DOUBLE_COLUMN_WIDTH * 0.8,
                    VisualizationConfig.SINGLE_COLUMN_WIDTH)
        )
        ax = ax[0]
        
        if train_steps is None:
            train_steps = list(range(len(train_losses)))
        
        # Plot training loss
        ax.plot(train_steps, train_losses, 
                color=IEEE_COLORS['blue'],
                linewidth=VisualizationConfig.LINE_WIDTH,
                marker='o', markersize=VisualizationConfig.MARKER_SIZE,
                label='Training Loss')
        
        # Plot validation loss
        ax.plot(train_steps, val_losses,
                color=IEEE_COLORS['red'],
                linewidth=VisualizationConfig.LINE_WIDTH,
                marker='s', markersize=VisualizationConfig.MARKER_SIZE,
                label='Validation Loss')
        
        # Configure plot
        ax.set_xlabel('Training Steps', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        ax.set_ylabel('Loss', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        ax.set_title('Training and Validation Loss Curves', 
                    fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                    fontweight='bold')
        ax.legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
        ax.grid(True, alpha=0.3)
        
        # Add grid lines
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
        
        return fig, ax
    
    @staticmethod
    def create_parameter_efficiency_plot(trainable_params, total_params, model_names):
        """Create parameter efficiency plot for LoRA"""
        fig, ax = IEEEVisualizer.create_figure(
            figsize=(VisualizationConfig.SINGLE_COLUMN_WIDTH,
                    VisualizationConfig.SINGLE_COLUMN_WIDTH * 0.8)
        )
        ax = ax[0]
        
        # Calculate efficiency percentage
        efficiency = [trainable/total*100 for trainable, total in zip(trainable_params, total_params)]
        
        # Create bar plot
        x = np.arange(len(model_names))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, total_params, width, 
                      label='Total Parameters', color=IEEE_COLORS['gray'], alpha=0.7)
        bars2 = ax.bar(x + width/2, trainable_params, width,
                      label='Trainable Parameters (LoRA)', color=IEEE_COLORS['blue'])
        
        # Add efficiency percentage text
        for i, eff in enumerate(efficiency):
            ax.text(i, trainable_params[i] * 1.05, f'{eff:.2f}%', 
                   ha='center', va='bottom', fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
        
        ax.set_xlabel('Model Configuration', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        ax.set_ylabel('Number of Parameters', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        ax.set_title('Parameter Efficiency with LoRA', 
                    fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                    fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=45, ha='right')
        ax.legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
        ax.set_yscale('log')
        
        return fig, ax
    
    @staticmethod
    def create_performance_comparison_plot(metrics_data):
        """Create performance comparison plot across models"""
        fig, axes = IEEEVisualizer.create_figure(nrows=1, ncols=2, 
            figsize=(VisualizationConfig.DOUBLE_COLUMN_WIDTH,
                    VisualizationConfig.SINGLE_COLUMN_WIDTH * 0.6))
        
        # Extract data
        models = list(metrics_data.keys())
        losses = [metrics_data[m]['loss'] for m in models]
        perplexities = [metrics_data[m].get('perplexity', 0) for m in models]
        colors = [IEEE_COLORS['blue'], IEEE_COLORS['red'], IEEE_COLORS['green']]
        
        # Plot 1: Loss comparison
        bars1 = axes[0].bar(models, losses, color=colors[:len(models)])
        axes[0].set_xlabel('Model', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[0].set_ylabel('Validation Loss', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[0].set_title('Validation Loss Comparison', 
                         fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                         fontweight='bold')
        
        # Add value labels
        for bar in bars1:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{height:.3f}', ha='center', va='bottom',
                        fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
        
        # Plot 2: Perplexity comparison
        if any(perplexities):
            bars2 = axes[1].bar(models, perplexities, color=colors[:len(models)])
            axes[1].set_xlabel('Model', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
            axes[1].set_ylabel('Perplexity', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
            axes[1].set_title('Perplexity Comparison', 
                             fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                             fontweight='bold')
            
            for bar in bars2:
                height = bar.get_height()
                axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                            f'{height:.2f}', ha='center', va='bottom',
                            fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
        
        return fig, axes
    
    @staticmethod
    def create_interactive_training_dashboard(train_metrics, val_metrics):
        """Create interactive Plotly dashboard for training analysis"""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Training Loss', 'Validation Loss', 
                           'Learning Rate Schedule', 'Gradient Norm'),
            vertical_spacing=0.15,
            horizontal_spacing=0.15
        )
        
        # Training Loss
        fig.add_trace(
            go.Scatter(x=list(range(len(train_metrics))), y=train_metrics,
                      mode='lines+markers', name='Training Loss',
                      line=dict(color=IEEE_COLORS['blue'], width=2),
                      marker=dict(size=4)),
            row=1, col=1
        )
        
        # Validation Loss
        fig.add_trace(
            go.Scatter(x=list(range(len(val_metrics))), y=val_metrics,
                      mode='lines+markers', name='Validation Loss',
                      line=dict(color=IEEE_COLORS['red'], width=2),
                      marker=dict(size=4)),
            row=1, col=2
        )
        
        # Update layout for IEEE style
        fig.update_layout(
            title="Training Analysis Dashboard",
            title_font=dict(size=16, family="Times New Roman"),
            showlegend=True,
            template="plotly_white",
            height=600,
            width=800
        )
        
        # Update axes
        fig.update_xaxes(title_text="Steps", title_font=dict(size=12))
        fig.update_yaxes(title_text="Loss", title_font=dict(size=12))
        
        # Save as HTML and static image
        html_path = os.path.join(VIZ_DIR, "training_dashboard.html")
        fig.write_html(html_path)
        
        png_path = os.path.join(VIZ_DIR, "training_dashboard.png")
        fig.write_image(png_path, width=1200, height=800)
        
        return fig

# Initialize visualizer
viz = IEEEVisualizer()
print("✅ IEEE Visualizer initialized")

## 4. Data Analysis and Visualization

Functions for analyzing the training data distribution, including
instruction/output length statistics and source composition.

In [ ]:
# Cell 5: Data Analysis and Visualization
def analyze_and_visualize_data(training_data):
    """Analyze training data and create visualizations"""
    print("\n📊 DATA ANALYSIS")
    print("="*50)
    
    # Create DataFrame for analysis
    df = pd.DataFrame(training_data)
    
    # Basic statistics
    print(f"Total examples: {len(df)}")
    
    # Length analysis
    if 'instruction' in df.columns and 'output' in df.columns:
        df['instruction_length'] = df['instruction'].str.len()
        df['output_length'] = df['output'].str.len()
        df['total_length'] = df['instruction_length'] + df['output_length']
        
        print(f"\nLength Statistics:")
        print(f"  Instruction length: {df['instruction_length'].mean():.1f} ± {df['instruction_length'].std():.1f}")
        print(f"  Output length: {df['output_length'].mean():.1f} ± {df['output_length'].std():.1f}")
        print(f"  Total length: {df['total_length'].mean():.1f} ± {df['total_length'].std():.1f}")
        
        # Create length distribution plot
        fig, axes = viz.create_figure(nrows=1, ncols=3, 
            figsize=(VisualizationConfig.DOUBLE_COLUMN_WIDTH,
                    VisualizationConfig.SINGLE_COLUMN_WIDTH * 0.8))
        
        # Plot 1: Instruction length distribution
        axes[0].hist(df['instruction_length'], bins=30, 
                    color=IEEE_COLORS['blue'], alpha=0.7, edgecolor='black')
        axes[0].axvline(df['instruction_length'].mean(), color='red', 
                       linestyle='--', linewidth=1.5, label=f'Mean: {df["instruction_length"].mean():.1f}')
        axes[0].set_xlabel('Instruction Length (chars)', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[0].set_ylabel('Frequency', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[0].set_title('Instruction Length Distribution', 
                         fontsize=VisualizationConfig.TITLE_FONT_SIZE)
        axes[0].legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
        
        # Plot 2: Output length distribution
        axes[1].hist(df['output_length'], bins=30, 
                    color=IEEE_COLORS['green'], alpha=0.7, edgecolor='black')
        axes[1].axvline(df['output_length'].mean(), color='red', 
                       linestyle='--', linewidth=1.5, label=f'Mean: {df["output_length"].mean():.1f}')
        axes[1].set_xlabel('Output Length (chars)', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[1].set_ylabel('Frequency', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[1].set_title('Output Length Distribution', 
                         fontsize=VisualizationConfig.TITLE_FONT_SIZE)
        axes[1].legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
        
        # Plot 3: Scatter plot of instruction vs output length
        scatter = axes[2].scatter(df['instruction_length'], df['output_length'],
                                 c=df['total_length'], cmap='viridis',
                                 alpha=0.6, edgecolors='black', linewidth=0.5)
        axes[2].set_xlabel('Instruction Length', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[2].set_ylabel('Output Length', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[2].set_title('Instruction vs Output Length', 
                         fontsize=VisualizationConfig.TITLE_FONT_SIZE)
        
        # Add colorbar
        cbar = fig.colorbar(scatter, ax=axes[2])
        cbar.set_label('Total Length', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        
        # Save the figure
        if TrainingConfig.SAVE_VISUALIZATIONS:
            viz.save_figure(fig, "data_length_distribution")
            print("✅ Data length distribution plot saved")
        plt.close(fig)
    
    # Source distribution (if available)
    if 'source' in df.columns:
        source_counts = df['source'].value_counts()
        print(f"\nData Sources:")
        for source, count in source_counts.items():
            print(f"  {source}: {count} ({count/len(df)*100:.1f}%)")
        
        # Create source distribution plot
        fig, ax = viz.create_figure()
        ax = ax[0]
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(source_counts)))
        wedges, texts, autotexts = ax.pie(source_counts.values, 
                                          labels=source_counts.index,
                                          autopct='%1.1f%%',
                                          colors=colors,
                                          startangle=90,
                                          textprops={'fontsize': VisualizationConfig.TICK_FONT_SIZE})
        
        ax.set_title('Data Source Distribution', 
                    fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                    fontweight='bold')
        
        # Improve autotext appearance
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
        
        # Save the figure
        if TrainingConfig.SAVE_VISUALIZATIONS:
            viz.save_figure(fig, "data_source_distribution")
            print("✅ Data source distribution plot saved")
        plt.close(fig)
    
    return df

# Continue with existing data loading...

## 5. Training Visualization Hooks

Callback class that collects training metrics (loss, learning rate,
gradient norms) during training for post-hoc visualization.

In [ ]:
# Cell 6: Enhanced Training with Visualization Hooks
class TrainingVisualizationCallback:
    """Callback for collecting training metrics for visualization"""
    
    def __init__(self):
        self.train_losses = []
        self.val_losses = []
        self.learning_rates = []
        self.gradient_norms = []
        self.steps = []
        self.current_step = 0
    
    def on_log(self, logs, step=None):
        """Collect metrics from training logs"""
        if step is None:
            step = self.current_step
            self.current_step += 1
        
        if 'loss' in logs:
            self.train_losses.append(logs['loss'])
            self.steps.append(step)
        
        if 'eval_loss' in logs:
            self.val_losses.append(logs['eval_loss'])
        
        if 'learning_rate' in logs:
            self.learning_rates.append(logs['learning_rate'])
        
        if 'grad_norm' in logs:
            self.gradient_norms.append(logs['grad_norm'])
    
    def create_training_summary(self):
        """Create comprehensive training summary visualizations"""
        if not self.train_losses:
            return None
        
        print("\n📈 TRAINING VISUALIZATION SUMMARY")
        print("="*50)
        
        # Create main training curves plot
        if len(self.train_losses) > 1 and len(self.val_losses) > 1:
            fig1, _ = viz.create_training_curves_plot(
                self.train_losses, 
                self.val_losses,
                self.steps[:len(self.train_losses)]
            )
            
            if TrainingConfig.SAVE_VISUALIZATIONS:
                viz.save_figure(fig1, "training_curves")
                print("✅ Training curves plot saved")
            
            plt.close(fig1)
        
        # Create learning rate schedule plot
        if self.learning_rates:
            fig2, ax = viz.create_figure()
            ax = ax[0]
            
            ax.plot(range(len(self.learning_rates)), self.learning_rates,
                   color=IEEE_COLORS['purple'],
                   linewidth=VisualizationConfig.LINE_WIDTH)
            
            ax.set_xlabel('Training Steps', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
            ax.set_ylabel('Learning Rate', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
            ax.set_title('Learning Rate Schedule', 
                        fontsize=VisualizationConfig.TITLE_FONT_SIZE,
                        fontweight='bold')
            ax.grid(True, alpha=0.3)
            
            if TrainingConfig.SAVE_VISUALIZATIONS:
                viz.save_figure(fig2, "learning_rate_schedule")
                print("✅ Learning rate schedule plot saved")
            
            plt.close(fig2)
        
        # Print summary statistics
        print(f"\nTraining Statistics:")
        print(f"  Final Training Loss: {self.train_losses[-1]:.4f}")
        if self.val_losses:
            print(f"  Final Validation Loss: {self.val_losses[-1]:.4f}")
            print(f"  Best Validation Loss: {min(self.val_losses):.4f}")
        print(f"  Total Training Steps: {len(self.train_losses)}")
        
        return {
            'final_train_loss': self.train_losses[-1] if self.train_losses else None,
            'final_val_loss': self.val_losses[-1] if self.val_losses else None,
            'best_val_loss': min(self.val_losses) if self.val_losses else None,
            'total_steps': len(self.train_losses)
        }

# Initialize callback
training_viz_callback = TrainingVisualizationCallback()

## 6. Model Evaluation with Visualizations

Enhanced evaluation functions that generate qualitative analysis
plots comparing model predictions against reference outputs.

In [ ]:
# Cell 7: Enhanced Model Evaluation with Visualizations
def evaluate_model_with_visualization(model, tokenizer, test_dataset, sample_count=10):
    """Evaluate model and create comprehensive visualizations"""
    print("\n📊 MODEL EVALUATION WITH VISUALIZATION")
    print("="*50)
    
    from transformers import pipeline
    
    # Create text generation pipeline
    generator = pipeline('text-generation', 
                        model=model, 
                        tokenizer=tokenizer,
                        device=0 if torch.cuda.is_available() else -1)
    
    results = {
        'predictions': [],
        'references': [],
        'bleu_scores': [],
        'rouge_scores': [],
        'perplexities': []
    }
    
    # Sample evaluation
    print(f"Evaluating on {min(sample_count, len(test_dataset))} samples...")
    
    for i in range(min(sample_count, len(test_dataset))):
        sample = test_dataset[i]
        
        if 'text' in sample:
            # Extract instruction from formatted text
            text = sample['text']
            if '<start_of_turn>user' in text:
                parts = text.split('<start_of_turn>user')[1].split('<end_of_turn>')
                instruction = parts[0].strip() if len(parts) > 0 else text
            elif '### Instruction:' in text:
                parts = text.split('### Instruction:')[1].split('### Response:')
                instruction = parts[0].strip() if len(parts) > 0 else text
            else:
                instruction = text[:100] + "..."
            
            # Generate response
            try:
                generated = generator(
                    instruction,
                    max_length=TrainingConfig.MAX_SEQ_LENGTH,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.95,
                    num_return_sequences=1
                )[0]['generated_text']
                
                results['predictions'].append(generated)
                results['references'].append(text)
                
            except Exception as e:
                print(f"  ⚠️ Error generating sample {i}: {e}")
    
    # Create qualitative analysis visualization
    if results['predictions']:
        create_qualitative_analysis_plot(results, test_dataset)
    
    return results

def create_qualitative_analysis_plot(results, test_dataset):
    """Create qualitative analysis plot for model outputs"""
    fig, axes = viz.create_figure(nrows=3, ncols=1, 
        figsize=(VisualizationConfig.DOUBLE_COLUMN_WIDTH,
                VisualizationConfig.DOUBLE_COLUMN_WIDTH * 0.8))
    
    # Plot 1: Response length comparison
    pred_lengths = [len(pred) for pred in results['predictions']]
    ref_lengths = [len(ref) for ref in results['references']]
    
    x = np.arange(len(pred_lengths))
    width = 0.35
    
    axes[0].bar(x - width/2, ref_lengths, width, 
               label='Reference', color=IEEE_COLORS['blue'], alpha=0.7)
    axes[0].bar(x + width/2, pred_lengths, width,
               label='Prediction', color=IEEE_COLORS['red'], alpha=0.7)
    
    axes[0].set_xlabel('Sample Index', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
    axes[0].set_ylabel('Text Length', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
    axes[0].set_title('Response Length Comparison', 
                     fontsize=VisualizationConfig.TITLE_FONT_SIZE)
    axes[0].legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Word frequency analysis (simple version)
    all_pred_text = ' '.join(results['predictions'])
    all_ref_text = ' '.join(results['references'])
    
    pred_words = all_pred_text.split()[:50]
    ref_words = all_ref_text.split()[:50]
    
    pred_word_freq = Counter(pred_words)
    ref_word_freq = Counter(ref_words)
    
    common_words = list(set(list(pred_word_freq.keys())[:10] + 
                           list(ref_word_freq.keys())[:10]))
    
    pred_freqs = [pred_word_freq.get(word, 0) for word in common_words]
    ref_freqs = [ref_word_freq.get(word, 0) for word in common_words]
    
    x = np.arange(len(common_words))
    axes[1].bar(x - width/2, ref_freqs, width, 
               label='Reference', color=IEEE_COLORS['blue'], alpha=0.7)
    axes[1].bar(x + width/2, pred_freqs, width,
               label='Prediction', color=IEEE_COLORS['red'], alpha=0.7)
    
    axes[1].set_xlabel('Common Words', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
    axes[1].set_ylabel('Frequency', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
    axes[1].set_title('Word Frequency Comparison', 
                     fontsize=VisualizationConfig.TITLE_FONT_SIZE)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(common_words, rotation=45, ha='right',
                           fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
    axes[1].legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
    
    # Plot 3: Sample responses table (simplified)
    axes[2].axis('off')
    sample_text = "Sample Predictions:\n\n"
    for i in range(min(3, len(results['predictions']))):
        sample_text += f"Sample {i+1}:\n"
        sample_text += f"Ref: {results['references'][i][:50]}...\n"
        sample_text += f"Pred: {results['predictions'][i][:50]}...\n\n"
    
    axes[2].text(0.05, 0.95, sample_text, 
                fontsize=VisualizationConfig.TICK_FONT_SIZE,
                verticalalignment='top',
                transform=axes[2].transAxes)
    
    # Save figure
    if TrainingConfig.SAVE_VISUALIZATIONS:
        viz.save_figure(fig, "qualitative_analysis")
        print("✅ Qualitative analysis plot saved")
    
    plt.close(fig)

## 7. IEEE Publication Report Generator

Generates a comprehensive report with LaTeX tables, JSON metrics,
and summary figures suitable for IEEE paper submissions.

In [ ]:
# Cell 8: Generate IEEE Publication Quality Report
def generate_ieee_report(training_metrics, evaluation_results, config):
    """Generate comprehensive IEEE-style report with all visualizations"""
    print("\n📑 GENERATING IEEE PUBLICATION REPORT")
    print("="*50)
    
    report_data = {
        'metadata': {
            'generation_date': datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            'model': config['model_id'],
            'training_config': {
                'epochs': TrainingConfig.NUM_EPOCHS,
                'learning_rate': TrainingConfig.LEARNING_RATE,
                'batch_size': TrainingConfig.BATCH_SIZE,
                'max_seq_length': TrainingConfig.MAX_SEQ_LENGTH,
                'lora_r': TrainingConfig.LORA_R,
                'lora_alpha': TrainingConfig.LORA_ALPHA
            }
        },
        'training_metrics': training_metrics,
        'evaluation_results': evaluation_results
    }
    
    # Save JSON report
    report_json_path = os.path.join(TrainingConfig.OUTPUT_DIR, "ieee_report.json")
    with open(report_json_path, 'w') as f:
        json.dump(report_data, f, indent=2)
    
    print(f"✅ JSON report saved: {report_json_path}")
    
    # Create LaTeX table for IEEE paper
    latex_table = create_latex_table(training_metrics, config)
    latex_path = os.path.join(TrainingConfig.OUTPUT_DIR, "results_table.tex")
    with open(latex_path, 'w') as f:
        f.write(latex_table)
    
    print(f"✅ LaTeX table saved: {latex_path}")
    
    # Create comprehensive summary figure
    create_summary_figure(training_metrics, evaluation_results, config)
    
    return report_data

def create_latex_table(training_metrics, config):
    """Create LaTeX table for IEEE paper"""
    latex = r"""\begin{table}[htbp]
\centering
\caption{Training Results for URA Tax Assistant Fine-tuning}
\label{tab:training_results}
\begin{tabular}{lccccc}
\toprule
\textbf{Metric} & \textbf{Value} & \textbf{Unit} & \textbf{Description} \\
\midrule
"""
    
    metrics = [
        ("Model", config['model_id'], "-", "Base model architecture"),
        ("LoRA Rank", TrainingConfig.LORA_R, "-", "LoRA rank parameter"),
        ("LoRA Alpha", TrainingConfig.LORA_ALPHA, "-", "LoRA scaling parameter"),
        ("Learning Rate", f"{TrainingConfig.LEARNING_RATE:.2e}", "-", "Initial learning rate"),
        ("Batch Size", TrainingConfig.BATCH_SIZE, "samples", "Per-device batch size"),
        ("Epochs", TrainingConfig.NUM_EPOCHS, "-", "Training epochs"),
        ("Final Train Loss", f"{training_metrics.get('final_train_loss', 'N/A'):.4f}", "-", "Final training loss"),
        ("Best Val Loss", f"{training_metrics.get('best_val_loss', 'N/A'):.4f}", "-", "Best validation loss"),
        ("Training Steps", training_metrics.get('total_steps', 'N/A'), "-", "Total training steps"),
    ]
    
    for metric, value, unit, desc in metrics:
        latex += f"{metric} & {value} & {unit} & {desc} \\\\\n"
    
    latex += r"""\bottomrule
\end{tabular}
\end{table}"""
    
    return latex

def create_summary_figure(training_metrics, evaluation_results, config):
    """Create summary figure combining all key results"""
    fig, axes = viz.create_figure(nrows=2, ncols=2,
        figsize=(VisualizationConfig.DOUBLE_COLUMN_WIDTH,
                VisualizationConfig.DOUBLE_COLUMN_WIDTH * 0.8))
    
    # Plot 1: Training summary
    metrics_to_plot = ['final_train_loss', 'best_val_loss']
    metric_names = ['Final Train Loss', 'Best Val Loss']
    values = [training_metrics.get(m, 0) for m in metrics_to_plot]
    
    axes[0, 0].bar(metric_names, values, 
                  color=[IEEE_COLORS['blue'], IEEE_COLORS['red']])
    axes[0, 0].set_ylabel('Loss Value', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
    axes[0, 0].set_title('Training Summary', 
                        fontsize=VisualizationConfig.TITLE_FONT_SIZE)
    
    # Add value labels
    for i, v in enumerate(values):
        axes[0, 0].text(i, v + 0.01, f'{v:.4f}', 
                       ha='center', va='bottom',
                       fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
    
    # Plot 2: Configuration parameters
    config_params = ['LORA_R', 'LORA_ALPHA', 'BATCH_SIZE', 'LEARNING_RATE']
    config_values = [TrainingConfig.LORA_R, TrainingConfig.LORA_ALPHA,
                    TrainingConfig.BATCH_SIZE, TrainingConfig.LEARNING_RATE]
    config_labels = ['LoRA Rank', 'LoRA Alpha', 'Batch Size', 'Learning Rate']
    
    x = np.arange(len(config_params))
    axes[0, 1].bar(x, config_values, color=IEEE_COLORS['purple'])
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(config_labels, rotation=45, ha='right',
                              fontsize=VisualizationConfig.TICK_FONT_SIZE-1)
    axes[0, 1].set_title('Configuration Parameters',
                        fontsize=VisualizationConfig.TITLE_FONT_SIZE)
    
    # Plot 3: Training steps distribution
    if 'train_losses' in training_metrics:
        steps = list(range(len(training_metrics['train_losses'])))
        axes[1, 0].plot(steps, training_metrics['train_losses'],
                       color=IEEE_COLORS['blue'], label='Training')
        if 'val_losses' in training_metrics:
            axes[1, 0].plot(steps[:len(training_metrics['val_losses'])], 
                          training_metrics['val_losses'],
                          color=IEEE_COLORS['red'], label='Validation')
        axes[1, 0].set_xlabel('Steps', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[1, 0].set_ylabel('Loss', fontsize=VisualizationConfig.AXIS_FONT_SIZE)
        axes[1, 0].set_title('Loss Curves', 
                            fontsize=VisualizationConfig.TITLE_FONT_SIZE)
        axes[1, 0].legend(fontsize=VisualizationConfig.LEGEND_FONT_SIZE)
    
    # Plot 4: Model info
    axes[1, 1].axis('off')
    model_info = f"""Model: {config['model_id']}
Target: {TrainingConfig.MODEL_TARGET}
Max Seq Length: {TrainingConfig.MAX_SEQ_LENGTH}
Parameters: {config.get('total_params', 'N/A'):,}
Trainable: {config.get('trainable_params', 'N/A'):,}
Trainable %: {config.get('trainable_pct', 0):.2f}%
Epochs: {TrainingConfig.NUM_EPOCHS}"""
    
    axes[1, 1].text(0.05, 0.95, model_info,
                   fontsize=VisualizationConfig.TICK_FONT_SIZE,
                   verticalalignment='top',
                   transform=axes[1, 1].transAxes,
                   family='monospace')
    
    # Adjust layout and save
    plt.suptitle('URA Tax Assistant Fine-tuning Summary', 
                fontsize=VisualizationConfig.TITLE_FONT_SIZE+2,
                fontweight='bold',
                y=1.02)
    
    if TrainingConfig.SAVE_VISUALIZATIONS:
        viz.save_figure(fig, "ieee_summary_figure")
        print("✅ IEEE summary figure saved")
    
    plt.close(fig)

## 8. Model Loading

Load the base model with Unsloth optimizations. Supports three modes:

1. **Fresh training**: Load base model and apply LoRA
2. **LoRA fine-tuning**: Load base model + pre-trained LoRA adapter
3. **Full model fine-tuning**: Load complete pre-trained model

In [ ]:
# Cell 8: Load Model with Unsloth Optimization
import gc
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("🚀 Loading model with Unsloth optimization...")

# Get model configuration
model_config = MODEL_CONFIGS[TrainingConfig.MODEL_TARGET]
model_id = model_config["model_id"]
max_seq_length = TrainingConfig.MAX_SEQ_LENGTH

# Check for pre-trained model from previous training
pretrained_model_path = "/kaggle/input/finetune_dataset/pretrained_model"
use_pretrained = False

if os.path.exists(pretrained_model_path):
    print(f"✅ Found pre-trained model at {pretrained_model_path}")
    
    # Check for LoRA adapter files
    adapter_files = [f for f in os.listdir(pretrained_model_path) if f.endswith(('.bin', '.safetensors')) and 'adapter' in f]
    
    if adapter_files:
        print("✅ Found LoRA adapter files, loading base model + adapter...")
        use_pretrained = True
        
        # Load base model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=True,
        )
        
        # Load LoRA adapter
        model = PeftModel.from_pretrained(model, pretrained_model_path)
        print("✅ Pre-trained LoRA adapter loaded")
        
    elif any(f.endswith('.bin') or f.endswith('.safetensors') for f in os.listdir(pretrained_model_path)):
        print("✅ Found full model files, loading complete model...")
        use_pretrained = True
        
        # Load the complete pre-trained model
        model = AutoModelForCausalLM.from_pretrained(
            pretrained_model_path,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            load_in_4bit=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(pretrained_model_path)
        print("✅ Complete pre-trained model loaded")
    else:
        print("⚠️ Pre-trained model directory exists but no valid model files found")
        
if not use_pretrained:
    print("ℹ️ No pre-trained model found, loading base model for fresh training...")
    # Load model with Unsloth (automatically applies optimizations)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        dtype=None,  # Auto-detect
        load_in_4bit=True,  # 4-bit quantization for memory efficiency
        # token = "hf_...", # Add your HF token if needed for gated models
    )

# Enable gradient checkpointing if specified
if TrainingConfig.USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")

# Prepare model for PEFT (LoRA) if not already loaded with adapter
if not use_pretrained or not hasattr(model, 'peft_config'):
    print("🔧 Applying LoRA configuration...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=TrainingConfig.LORA_R,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj",],
        lora_alpha=TrainingConfig.LORA_ALPHA,
        lora_dropout=TrainingConfig.LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing=TrainingConfig.USE_GRADIENT_CHECKPOINTING,
        random_state=SEED,
        use_rslora=False,  # Use standard LoRA
        loftq_config=None,  # No LoftQ initialization
    )
    print("✅ LoRA configuration applied")

# Configure tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n✅ Model loaded and optimized")
if use_pretrained:
    print(f"   Mode: Fine-tuning pre-trained model")
else:
    print(f"   Mode: Training from base model")
print(f"   Base Model: {model_id}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Free up memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 9. Dataset Loading and Preparation

Load teacher QA data from JSONL format, format for Gemma chat template,
split into train/test sets, and create HuggingFace datasets.

In [ ]:
# Cell 9: Load Teacher QA Dataset

# Define data paths
TEACHER_QA_PATH = "/kaggle/input/finetune_dataset/teacher_qa.jsonl"  # Kaggle path
LOCAL_TEACHER_QA_PATH = "/home/darkhorse/FinalYearProject/Data/teacher_qa/teacher_qa.jsonl"  # Local path

def load_teacher_qa_data():
    """Load teacher QA data from JSONL file"""
    data = []
    
    # Try Kaggle path first, then local path
    teacher_qa_file = None
    if Path(TEACHER_QA_PATH).exists():
        teacher_qa_file = TEACHER_QA_PATH
    elif Path(LOCAL_TEACHER_QA_PATH).exists():
        teacher_qa_file = LOCAL_TEACHER_QA_PATH
    else:
        print("⚠️ Teacher QA data not found. Using synthetic data for demonstration.")
        return create_synthetic_data()
    
    print(f"📂 Loading teacher QA data from: {teacher_qa_file}")
    
    with open(teacher_qa_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                item = json.loads(line.strip())
                data.append(item)
            except json.JSONDecodeError as e:
                print(f"⚠️ Skipping invalid JSON at line {line_num}: {e}")
                continue
    
    print(f"✅ Loaded {len(data)} teacher QA samples")
    
    # Show sample data structure
    if data:
        sample = data[0]
        print(f"   Sample fields: {list(sample.keys())}")
        print(f"   Question types: {set(item.get('question_type', 'unknown') for item in data[:100])}")
    
    return data

def create_synthetic_data():
    """Create synthetic data for demonstration when teacher QA is not available"""
    print("🔧 Creating synthetic training data...")
    
    synthetic_data = [
        {
            "question": "What is the capital of France?",
            "answer": "The capital of France is Paris.",
            "source": "synthetic",
            "question_type": "factual"
        },
        {
            "question": "How does photosynthesis work?",
            "answer": "Photosynthesis is the process by which plants convert sunlight, carbon dioxide, and water into glucose and oxygen.",
            "source": "synthetic", 
            "question_type": "explanatory"
        },
        {
            "question": "What are the main tax obligations for businesses in Uganda?",
            "answer": "Businesses in Uganda must register for taxes, file returns, and pay various taxes including corporate income tax, VAT, and withholding tax.",
            "source": "ura_tax_guide",
            "question_type": "procedural"
        }
    ] * 100  # Repeat to create more samples
    
    print(f"✅ Created {len(synthetic_data)} synthetic samples")
    return synthetic_data

def format_for_training(data):
    """Format data for Gemma training"""
    formatted_data = []
    
    for item in data:
        question = item.get('question', '')
        answer = item.get('answer', '')
        source = item.get('source', 'unknown')
        question_type = item.get('question_type', 'unknown')
        
        # Format for Gemma chat template
        formatted_text = f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>"
        
        formatted_data.append({
            'text': formatted_text,
            'question': question,
            'answer': answer,
            'source': source,
            'question_type': question_type
        })
    
    return formatted_data

# Load and format teacher QA data
print("📚 Loading Teacher QA Dataset")
print("="*50)

raw_teacher_data = load_teacher_qa_data()
training_data = format_for_training(raw_teacher_data)

print(f"✅ Formatted {len(training_data)} training examples")

# Create train/test split
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    training_data, 
    test_size=TrainingConfig.TRAIN_TEST_SPLIT, 
    random_state=SEED,
    stratify=[item['question_type'] for item in training_data] if len(training_data) > 0 else None
)

print(f"📊 Data split: {len(train_data)} train, {len(test_data)} test")

# Convert to HuggingFace dataset format
from datasets import Dataset

dataset_dict = {
    "train": Dataset.from_list(train_data),
    "test": Dataset.from_list(test_data)
}

print("✅ Dataset prepared for training")
print(f"   Train samples: {len(dataset_dict['train'])}")
print(f"   Test samples: {len(dataset_dict['test'])}")

# Analyze data
df = analyze_and_visualize_data(training_data)

## 10. Tokenization

Tokenize the formatted dataset using the model's tokenizer with
truncation and padding to `MAX_SEQ_LENGTH`.

In [ ]:
# Cell 10: Tokenization Function
def tokenize_function(examples):
    """Tokenize the examples"""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=TrainingConfig.MAX_SEQ_LENGTH,
        return_tensors=None,
    )

print("🔤 Tokenizing dataset...")

# Tokenize dataset
tokenized_dataset = dataset_dict.map(
    tokenize_function,
    batched=True,
    num_proc=2,
    remove_columns=dataset_dict["train"].column_names,
)

print(f"✅ Tokenization complete")
print(f"   Example token length: {len(tokenized_dataset['train'][0]['input_ids'])}")

## 11. Training Configuration

Configure HuggingFace `TrainingArguments` with optimized settings for Kaggle:
paged AdamW 8-bit optimizer, cosine LR schedule, mixed precision, and
gradient checkpointing.

In [ ]:
# Cell 11: Training Configuration Setup
# Calculate effective batch size
effective_batch_size = TrainingConfig.BATCH_SIZE * TrainingConfig.GRADIENT_ACCUMULATION
total_steps = (len(tokenized_dataset["train"]) // effective_batch_size) * TrainingConfig.NUM_EPOCHS
warmup_steps = min(TrainingConfig.WARMUP_STEPS, total_steps // 10)

print(f"\n📊 Training Statistics:")
print(f"   Total training steps: {total_steps}")
print(f"   Warmup steps: {warmup_steps}")
print(f"   Effective batch size: {effective_batch_size}")

# Training arguments optimized for Kaggle
training_args = TrainingArguments(
    output_dir=TrainingConfig.OUTPUT_DIR,
    num_train_epochs=TrainingConfig.NUM_EPOCHS,
    per_device_train_batch_size=TrainingConfig.BATCH_SIZE,
    per_device_eval_batch_size=TrainingConfig.BATCH_SIZE,
    gradient_accumulation_steps=TrainingConfig.GRADIENT_ACCUMULATION,
    learning_rate=TrainingConfig.LEARNING_RATE,
    warmup_steps=warmup_steps,
    logging_steps=TrainingConfig.LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=TrainingConfig.EVAL_STEPS,
    save_strategy="steps",
    save_steps=TrainingConfig.SAVE_STEPS,
    save_total_limit=TrainingConfig.SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    report_to=["tensorboard"] if TrainingConfig.LOGGING_STEPS > 0 else [],
    
    # Optimization
    optim="paged_adamw_8bit",  # Optimized optimizer for 8-bit weights
    gradient_checkpointing=TrainingConfig.USE_GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Mixed precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Regularization
    max_grad_norm=TrainingConfig.MAX_GRAD_NORM,
    weight_decay=TrainingConfig.WEIGHT_DECAY,
    
    # Other
    dataloader_num_workers=2,
    group_by_length=TrainingConfig.PACKING,
    lr_scheduler_type="cosine",
    seed=SEED,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
)

## 12. Initialize Trainer

Initialize the SFTTrainer with the model, tokenized datasets,
and training arguments.

In [ ]:
# Cell 12: Initialize Trainer
print("🚀 Starting training...")
print("="*60)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=TrainingConfig.MAX_SEQ_LENGTH,
    packing=TrainingConfig.PACKING,  # Pack sequences for efficiency
    
    # Optional: Use NEFTune for better generalization
    # neftune_noise_alpha=5.0,
)

## 13. Execute Training

Run the training loop with integrated visualization metric collection.

In [ ]:
# Cell 13: Execute Training
print("\n\U0001f680 STARTING TRAINING")
print("=" * 60)

# Train the model
train_result = trainer.train()

# Collect training metrics
training_metrics = train_result.metrics
print(f"\n\u2705 Training complete!")
print(f"   Total steps: {training_metrics.get('train_steps', 'N/A')}")
print(f"   Training loss: {training_metrics.get('train_loss', 'N/A'):.4f}")
print(f"   Runtime: {training_metrics.get('train_runtime', 0):.1f}s")

# Feed metrics to visualization callback
for log_entry in trainer.state.log_history:
    training_viz_callback.on_log(log_entry)

# Generate training summary visualizations
training_summary = training_viz_callback.create_training_summary()

## 14. Model Evaluation

Evaluate the trained model on the held-out test set.

In [ ]:
# Cell 14: Evaluation
# Evaluate the model
print("📊 Evaluating model...")
eval_results = trainer.evaluate()

print("\n📈 Evaluation Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

## 15. Export Visualizations and Report

Generate the IEEE report and export all visualization artifacts.

In [ ]:
# Cell 15: Generate IEEE Report and Export Visualizations
import shutil

# Generate IEEE report with actual training data
model_info = {
    'model_id': MODEL_CONFIGS[TrainingConfig.MODEL_TARGET]['model_id'],
    'total_params': sum(p.numel() for p in model.parameters()),
    'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
    'trainable_pct': sum(p.numel() for p in model.parameters() if p.requires_grad) /
                     sum(p.numel() for p in model.parameters()) * 100
}

report_data = generate_ieee_report(
    training_summary if training_summary else {},
    eval_results,
    model_info
)

# Export all visualizations to IEEE directory
export_dir = "/kaggle/working/ieee_figures"
os.makedirs(export_dir, exist_ok=True)

viz_files = []
for root, dirs, files in os.walk(VIZ_DIR):
    for file in files:
        if file.endswith(('.png', '.pdf', '.svg')):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(export_dir, file)
            shutil.copy2(src_path, dst_path)
            viz_files.append(dst_path)

print(f"\u2705 Exported {len(viz_files)} visualization files to {export_dir}")

## 16. Test Inference

Test the fine-tuned model with a sample tax question to verify quality.

In [ ]:
# Cell 16: Test Inference
# Test inference
print("\n🤖 Testing inference with a sample question...")

# Prepare a test prompt
test_prompt = """What is the property tax rate for residential properties in Singapore?"""

# Format based on model type
if "gemma" in TrainingConfig.MODEL_TARGET.lower():
    formatted_prompt = f"<start_of_turn>user\n{test_prompt}<end_of_turn>\n<start_of_turn>model\n"
elif "llama" in TrainingConfig.MODEL_TARGET.lower():
    formatted_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{test_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
else:
    formatted_prompt = f"### Instruction:\n{test_prompt}\n\n### Response:\n"

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=512)

if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the assistant's response
if "gemma" in TrainingConfig.MODEL_TARGET.lower():
    # Extract text after "model" marker
    if "<start_of_turn>model" in response:
        response = response.split("<start_of_turn>model")[1].strip()
elif "llama" in TrainingConfig.MODEL_TARGET.lower():
    if "assistant" in response:
        response = response.split("assistant")[1].strip()

print(f"\n💬 Model Response:")
print(response[:500] + "..." if len(response) > 500 else response)

## 17. Save to Kaggle Output

Create a zip archive of the fine-tuned model for Kaggle output.

In [ ]:
# Cell 17: Save to Kaggle Output
# Create a zip file for Kaggle output
import shutil

output_zip = "/kaggle/working/finetuned_model.zip"

print(f"\n📦 Creating output archive...")
shutil.make_archive(
    "/kaggle/working/finetuned_model",
    'zip',
    TrainingConfig.OUTPUT_DIR
)

print(f"✅ Archive created: {output_zip}")
print(f"   Size: {os.path.getsize(output_zip) / 1024 / 1024:.2f} MB")

# Clean up to save space (optional)
!rm -rf {TrainingConfig.OUTPUT_DIR}

## 18. Advanced Optimizations

Optional post-training optimizations including Flash Attention 2
and gradient checkpointing configuration.

In [ ]:
# Cell 18: Advanced Optimizations
# 1. Use Flash Attention 2 (if not already enabled)
if TrainingConfig.USE_FLASH_ATTENTION_2:
    try:
        model.config._attn_implementation = "flash_attention_2"
        print("✅ Flash Attention 2 enabled")
    except (AttributeError, ImportError, RuntimeError) as e:
        print(f"⚠️  Flash Attention 2 not available: {e}")

# 2. Use gradient checkpointing with custom settings
if TrainingConfig.USE_GRADIENT_CHECKPOINTING:
    # Enable selective gradient checkpointing
    model.enable_input_require_grads()
    model.config.use_cache = False  # Disable cache for gradient checkpointing


## 19. Monitoring and Logging

GPU memory monitoring and comprehensive training report generation.

In [ ]:
# Cell 19: Monitoring & Logging Functions
# Monitor GPU usage during training
def monitor_gpu():
    if torch.cuda.is_available():
        print(f"\n📊 GPU Memory Usage:")
        print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"   Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
        print(f"   Max Allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        torch.cuda.reset_peak_memory_stats()

# Create training report
def create_training_report():
    """Create a comprehensive training report"""
    report = {
        "model": TrainingConfig.MODEL_TARGET,
        "base_model": model_config["model_id"],
        "training_date": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "training_duration": training_metrics.get("train_runtime", 0) if "training_metrics" in dir() else 0,
        "samples": len(dataset_dict["train"]),
        "validation_samples": len(dataset_dict["test"]),
        "hyperparameters": {
            "learning_rate": TrainingConfig.LEARNING_RATE,
            "batch_size": TrainingConfig.BATCH_SIZE,
            "epochs": TrainingConfig.NUM_EPOCHS,
            "max_seq_length": TrainingConfig.MAX_SEQ_LENGTH,
            "lora_r": TrainingConfig.LORA_R,
            "lora_alpha": TrainingConfig.LORA_ALPHA,
        },
        "evaluation": eval_results,
        "optimizations": {
            "unsloth": True,
            "qlora": True,
            "gradient_checkpointing": TrainingConfig.USE_GRADIENT_CHECKPOINTING,
            "flash_attention": TrainingConfig.USE_FLASH_ATTENTION_2,
        }
    }
    
    report_path = os.path.join(TrainingConfig.OUTPUT_DIR, "training_report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    
    print(f"✅ Training report saved to: {report_path}")
    return report

## 20. Final Report

Generate the final training report and display execution summary.

In [ ]:
# Cell 20: Generate Final Report
# Generate report
monitor_gpu()
report = create_training_report()

print("\n" + "="*60)
print("✅ NOTEBOOK EXECUTION COMPLETE")
print("="*60)
print(f"\nMode: {'Fine-tuning' if TrainingConfig.IS_FINETUNING else 'Initial Training'}")
print(f"Model: {TrainingConfig.MODEL_TARGET}")
print(f"Output: {TrainingConfig.OUTPUT_DIR}")
print(f"Training Examples: {len(dataset_dict['train'])}")
print(f"Validation Loss: {eval_results.get('eval_loss', 'N/A'):.4f}")
if TrainingConfig.IS_FINETUNING:
    print(f"Pre-trained Model: ✅ Used from {TrainingConfig.PRETRAINED_MODEL_PATH}")

## Summary

This notebook provides a highly optimized fine-tuning pipeline for Kaggle with:

1. **Unsloth Integration**: 2x faster training with automatic optimizations
2. **QLoRA**: 4-bit quantization for memory efficiency
3. **Flash Attention**: Faster attention computation
4. **Optimized Training Loop**: With proper batching and accumulation
5. **Fine-tuning Support**: Can load and further fine-tune pre-trained models
6. **Automatic Mode Detection**: Adjusts parameters based on training vs fine-tuning
7. **IEEE Visualization**: Publication-quality charts and metrics

### Key Features

| Feature | Description |
|---------|-------------|
| Adaptive Training | Automatically detects pre-trained models and adjusts parameters |
| Memory Efficient | 4-bit quantization with gradient checkpointing |
| Fast Training | Unsloth optimizations for 2x speed improvement |
| Visualization | Comprehensive metrics and IEEE publication charts |
| Kaggle Integration | Proper output formatting and zipping |

### Usage Modes

- **Initial Training**: Load base model and train from scratch
- **Fine-tuning**: Load pre-trained model and further optimize

### Execution Order

1. Run cells 1-4 for setup and configuration
2. Cells 5-7 define visualization utilities (auto-executed)
3. Cell 8 loads the model
4. Cell 9 loads and prepares training data
5. Cells 10-13 configure and run training
6. Cells 14-17 evaluate, export, test inference, and save
7. Cells 18-20 apply optimizations and generate final report